## Optimise model

In [ ]:
!python optimize_model.py \
      --model yolo26s.pt \
      --formats onnx,tflite,openvino,ncnn,engine


Initializing YOLOv8 Model Optimizer
  Input Model: yolo26s.pt
  Image Size:  640
  FP16 (Half): False
  INT8 Quant:  False

Loading PyTorch model weights...

--- Exporting to 'ONNX' format ---
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26s summary (fused): 122 layers, 9,496,140 parameters, 0 gradients, 20.7 GFLOPs

PyTorch: starting from 'yolo26s.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (19.5 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 360ms
Prepared 4 packages in 1.46s
Installed 4 packages in 309ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime==1.26.0
 + onnxslim==0.1.94

requirements: AutoUpdate success

In [ ]:
!ls

calibration_image_sample_data_20x128x128x3_float32.npy	yolo26s.onnx
drive							yolo26s_openvino_model
optimize_model.py					yolo26s.pt
sample_data						yolo26s_saved_model
yolo26s_ncnn_model


In [ ]:
!nvidia-smi

Fri May 22 12:37:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
torch.cuda.is_available()

True

# Optimizing & Benchmarking New Models

In [ ]:
!python optimize_model.py --model model/best.pt --formats all


Initializing YOLOv8 Model Optimizer
  Input Model: model/best.pt
  Image Size:  640
  Dynamic:     True
  Simplify:    True
  Force Flag:  False
  Total Runs Scheduled: 12

Loading PyTorch model weights...

--- Exporting to 'ONNX' (fp32) format ---
  Export parameters: {'format': 'onnx', 'imgsz': 640, 'half': False, 'dynamic': True, 'simplify': True}
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from 'model/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (21.5 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 3.3s, saved as 'model/best.onnx' (42.9 MB)

Export complete (4.7s)
Results saved to /content/model/best.onnx
Predict:         yolo predict task=detect model=model/best.onnx imgsz=640 
Validate:        yolo val task=detect model=model/best.onn

In [ ]:
# !cp model/best.pt best.pt && rm -rf model/* && cp best.pt model/

In [ ]:
!python optimize_model.py --model model/best.pt --formats all


Initializing YOLOv8 Model Optimizer
  Input Model: model/best.pt
  Image Size:  640
  Dynamic:     True
  Simplify:    True
  Force Flag:  False
  Total Runs Scheduled: 12

Loading PyTorch model weights...

--- Skipping 'ONNX' (fp32) ---
  Model already exists at: model/onnx/fp32/best.onnx
  Skipping export. Use --force to overwrite.

--- Skipping 'ONNX' (fp16) ---
  Model already exists at: model/onnx/fp16/best.onnx
  Skipping export. Use --force to overwrite.

--- Exporting to 'ENGINE' (fp32) format ---
  Export parameters: {'format': 'engine', 'imgsz': 640, 'half': False, 'int8': False, 'dynamic': True, 'simplify': True, 'device': 0}
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
WARNING ⚠️ 'dynamic=True' model with 'format=engine' requires max batch size, i.e. 'batch=16'
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from 'model/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape

In [ ]:
#Dynamic Batch
!python benchmark_edge.py --model-dir model/

Discovered 18 models in 'model/':
  - model/best.onnx
  - model/best.pt
  - model/best_saved_model/best_float16.tflite
  - model/best_saved_model/best_float32.tflite
  - model/best_saved_model/best_full_integer_quant.tflite
  - model/best_saved_model/best_integer_quant.tflite
  - model/ncnn/fp16/best_ncnn_model
  - model/ncnn/fp32/best_ncnn_model
  - model/onnx/fp16/best.onnx
  - model/onnx/fp32/best.onnx
  - model/openvino/fp16/best_openvino_model
  - model/openvino/fp32/best_openvino_model
  - model/openvino/int8/best_int8_openvino_model
  - model/tensorrt/fp16/best.engine
  - model/tensorrt/fp32/best.engine
  - model/tensorrt/int8/best.engine
  - model/tflite/fp32/best_float32.tflite
  - model/tflite/int8/best_int8.tflite

Benchmarking Model: model/best.onnx
Model File Size: 42.76 MB
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Running warmup frames...
Lo

In [ ]:
import shutil
shutil.make_archive('models', 'zip', 'model')

'/content/models.zip'

In [ ]:
import shutil
shutil.make_archive('models', 'zip', 'model')

In [ ]:
from IPython.core.display import json
!python benchmark_edge.py \
--model-dir model \
--save-json benchmark_results/results.json \
--save-csv benchmark_results/results.csv

Discovered 18 models in 'model':
  - model/best.onnx
  - model/best.pt
  - model/best_saved_model/best_float16.tflite
  - model/best_saved_model/best_float32.tflite
  - model/best_saved_model/best_full_integer_quant.tflite
  - model/best_saved_model/best_integer_quant.tflite
  - model/ncnn/fp16/best_ncnn_model
  - model/ncnn/fp32/best_ncnn_model
  - model/onnx/fp16/best.onnx
  - model/onnx/fp32/best.onnx
  - model/openvino/fp16/best_openvino_model
  - model/openvino/fp32/best_openvino_model
  - model/openvino/int8/best_int8_openvino_model
  - model/tensorrt/fp16/best.engine
  - model/tensorrt/fp32/best.engine
  - model/tensorrt/int8/best.engine
  - model/tflite/fp32/best_float32.tflite
  - model/tflite/int8/best_int8.tflite

Benchmarking Model: model/best.onnx
Model File Size: 42.76 MB
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Running warmup frames...
Loa

In [ ]:
import shutil
shutil.make_archive('benchmark_results', 'zip', 'benchmark_results')

'/content/benchmark_results.zip'